# Week 10 Post-Class: Solutions

Reference solution for Part B (fine-tuning DistilBERT on IMDB).

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
    pipeline,
)
from datasets import load_dataset
import numpy as np

set_seed(42)

# Load IMDB sentiment dataset
ds = load_dataset('imdb')
train_ds = ds['train'].shuffle(seed=42).select(range(2000))
test_ds = ds['test'].shuffle(seed=42).select(range(500))
print(f'Train: {len(train_ds)}, Test: {len(test_ds)}')
print(f'Classes: {train_ds.features["label"].names}')

In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, max_length=256, padding='max_length')

train_tokenized = train_ds.map(tokenize, batched=True)
test_tokenized = test_ds.map(tokenize, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir='./distilbert_imdb',
    num_train_epochs=3,
    per_device_train_batch_size=8,  # Smaller batch for longer max_length
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    eval_strategy='epoch',
    logging_steps=50,
    save_strategy='no',
    report_to='none',
    disable_tqdm=True,   # plain-text progress: avoids a transformers v5 notebook
                         # progress-bar callback bug and the ipywidgets dependency
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    return {'accuracy': (np.argmax(predictions, axis=1) == labels).mean()}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())

In [ ]:
# Use the fine-tuned model on new examples
label_names = ['negative', 'positive']

# Use the model's OWN device so inputs/weights match on CPU, CUDA, or Apple MPS.
device = next(model.parameters()).device

examples = [
    'This movie was a complete waste of time. Terrible acting and pointless plot.',
    'Absolutely loved every minute. The cinematography was breathtaking.',
    "It's not great, but not awful either. Watchable on a slow afternoon.",
]

for text in examples:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()
    pred = label_names[probs.argmax()]
    print(f'[{pred:8s}] ({probs.max():.3f}) {text}')

## Typical reflection answers

1. **Test accuracy:** ~88-92% on IMDB with 2K training examples. With full 25K training, would reach ~94%.

2. **Surprises:**
   - Easier than expected — fine-tuning is simple
   - Model handles negation well ("not great" → negative)
   - Mixed-sentiment reviews often classified by the dominant sentiment

3. **2015 comparison:** Would have required:
   - Word embeddings (Word2Vec or GloVe — train or download)
   - Custom architecture (LSTM or CNN)
   - Hand-tuned hyperparameters
   - 10× more data to get comparable accuracy
   - Days of training instead of minutes

4. **Next improvements:**
   - More data (always)
   - Try RoBERTa or DeBERTa-base instead of DistilBERT
   - Longer max_length (256 → 512) for full-length reviews
   - Different learning rate schedule (warm-up + decay)